In [7]:
import gymnasium as gym
import highway_env
import numpy as np
from stable_baselines3 import DQN, PPO, A2C  # Importa las clases que use tu equipo

# 1. Configuración del entorno de evaluación estándar oficial[cite: 4]
EVAL_ENV_NAME = 'highway-v0'
eval_env = gym.make(EVAL_ENV_NAME, render_mode='rgb_array')

# 2. Aplicar los mismos wrappers si vuestro equipo utilizó alguno[cite: 4]
def aplicar_wrappers(env):
    # env = MiWrapper(env)
    return env

eval_env = aplicar_wrappers(eval_env)

# 3. Diccionario con los modelos del equipo que quieres comparar
# Formato: "Nombre_Modelo": (Clase_Algoritmo, "ruta/al/best_model.zip")
modelos_equipo = {
    #"DQN": (DQN, "logs/DQN/best_model.zip"),
    "PPO": (PPO, "logs/PPO/best_model.zip")
    #"A2C": (A2C, "logs/A2C/best_model.zip"),
}

# 4. Configuración fija oficial de evaluación[cite: 4]
EVAL_EPISODES = 20
TEST_SEED = 200
TEST_SEEDS = [TEST_SEED + i for i in range(EVAL_EPISODES)]

resultados_equipo = {}

# 5. Bucle para recorrer y evaluar todos los modelos de forma idéntica al test oficial
for nombre, (Algoritmo, ruta_modelo) in modelos_equipo.items():
    print(f"\nEvaluando a: {nombre}...")
    try:
        # Carga del agente vinculándolo al entorno de evaluación[cite: 4]
        agent = Algoritmo.load(ruta_modelo, env=eval_env)
        
        episode_rewards = []
        episode_mean_speeds = []
        episode_collisions = []
        episode_lengths = []

        # Ejecución de los 20 episodios con las semillas reservadas del test[cite: 4]
        for episode_seed in TEST_SEEDS:
            state, info = eval_env.reset(seed=episode_seed)
            terminated = truncated = False

            episode_reward = 0.0
            speeds = []
            collided = False
            steps = 0

            while not (terminated or truncated):
                action, _ = agent.predict(state, deterministic=True)
                state, reward, terminated, truncated, info = eval_env.step(action)

                episode_reward += float(reward)
                speeds.append(float(info["speed"]))  # Recoge velocidad oficial[cite: 4]
                collided = collided or bool(info["crashed"])  # Registra colisión oficial[cite: 4]
                steps += 1

            episode_rewards.append(episode_reward)
            episode_mean_speeds.append(float(np.mean(speeds)))
            episode_collisions.append(collided)
            episode_lengths.append(steps)

        # Cálculo de las métricas oficiales[cite: 4]
        mean_reward = np.mean(episode_rewards)
        std_reward = np.std(episode_rewards)
        mean_speed = np.mean(episode_mean_speeds)
        std_speed = np.std(episode_mean_speeds)
        collisions = int(np.sum(episode_collisions))
        collision_rate = 100 * collisions / EVAL_EPISODES
        mean_length = np.mean(episode_lengths)

        # Guardar resultados en el diccionario
        resultados_equipo[nombre] = {
            "mean_reward": mean_reward,
            "std_reward": std_reward,
            "mean_speed": mean_speed,
            "collisions": collisions,
            "collision_rate": collision_rate,
            "mean_length": mean_length
        }

        print(f"-> {nombre} evaluado con éxito. Recompensa: {mean_reward:.2f} ± {std_reward:.2f}")

    except Exception as e:
        print(f"❌ Error al cargar o evaluar {nombre}. Revisa la ruta o el algoritmo. Detalle: {e}")

eval_env.close()

# 6. Mostrar el ranking final detallado del equipo
if resultados_equipo:
    print("\n" + "="*50)
    print("🏆 RANKING FINAL DEL EQUIPO (Criterios oficiales) 🏆")
    print("="*50)
    
    # Ordenar por recompensa media de mayor a menor (criterio principal de la competición)[cite: 4]
    ranking = sorted(resultados_equipo.items(), key=lambda x: x[1]["mean_reward"], reverse=True)
    
    for pos, (nombre, m) in enumerate(ranking, 1):
        print(f"{pos}. **{nombre}**")
        print(f"   - Recompensa media: {m['mean_reward']:.2f} ± {m['std_reward']:.2f}")
        print(f"   - Velocidad media: {m['mean_speed']:.2f} m/s")
        print(f"   - Episodios con colisión: {m['collisions']} de {EVAL_EPISODES} ({m['collision_rate']:.1f} %)")
        print(f"   - Longitud media: {m['mean_length']:.1f} pasos")
        print("-" * 45)
        
    mejor_modelo_equipo = ranking[0][0]
    print(f"\n👉 Conclusión: El mejor modelo para entregar es **{mejor_modelo_equipo}**")
    print("="*50)


Evaluando a: PPO...
-> PPO evaluado con éxito. Recompensa: 23.32 ± 9.18

🏆 RANKING FINAL DEL EQUIPO (Criterios oficiales) 🏆
1. **PPO**
   - Recompensa media: 23.32 ± 9.18
   - Velocidad media: 20.68 m/s
   - Episodios con colisión: 4 de 20 (20.0 %)
   - Longitud media: 33.4 pasos
---------------------------------------------

👉 Conclusión: El mejor modelo para entregar es **PPO**
